# Libs

In [ ]:
import sys
sys.path.append("../libs/")
sys.path.append("../")

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

import futurai_ppd as ppd
import futurai_utils as utils
from futurai_ml_dev import FuturaiML

DIR_DATA = os.getcwd() + "/data/"

# Load Data

In [ ]:
process_name = "Refinador - Rosca Extração"
timestamp="TIMESTAMP"

list_colomuns_drop = ["espessura_receita_prensa"]

df_dataset = ppd.load_dataset_principal(DIR_DATA+process_name+".csv", list_colomuns_drop, timestamp, dropna=True, use_chunks=True, chunksize=10000)
df_dataset

# Set ON/OFF var

In [ ]:
pre_process = []
pp_var_ref_desligado = "R_2314S_Motor"
pp_valor_ref_desligado = 400
pp_tempo_ref_desligado = 0
pp_pre_corte_transitorio = 60
pp_pos_corte_transitorio = 210
pre_process.append(  
{
   "after_cut": pp_pos_corte_transitorio,
   "interval_off": pp_tempo_ref_desligado,
   "limit_off": pp_valor_ref_desligado,
   "pre_cut": pp_pre_corte_transitorio,
   "variable_off": pp_var_ref_desligado
  })

# Preprocessing Data

In [ ]:
print(f"Dataset shape before preprocessing: {df_dataset.shape}")
for pro in pre_process:
    df_dataset,_,_ = ppd.drop_transitorio_desligado(df_dataset,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
df_dataset.reset_index(inplace=True, drop=True)
print(f"Dataset shape after preprocessing: {df_dataset.shape}")

# TAGs and descriptions

In [ ]:
df_sistema, df_sistema_drop =  ppd.set_tags_config(df_dataset,DIR_DATA+process_name+"_subsistema.csv")
df_sistema

# Select training periods (Atual)

In [ ]:
fig = utils.select_training_period(df_dataset, timestamp)
fig.show()

# Data Train

## Period 1

In [ ]:
start_date_train = pd.to_datetime("2025-07-10 00:00:00")
end_date_train = pd.to_datetime("2025-07-15 00:00:00")

mask = (df_dataset[timestamp] >= start_date_train) & (df_dataset[timestamp] <= end_date_train)
df_train = df_dataset.loc[mask]

eixoX_train = df_train[timestamp]
df_train = df_train.drop(timestamp,axis=1)

## Period 2

In [ ]:
start_date_train2 = pd.to_datetime("2025-10-21 20:00:00")
end_date_train2 = pd.to_datetime("2025-10-23 09:00:00")

mask = (df_dataset[timestamp] > start_date_train2) & (df_dataset[timestamp] <= end_date_train2)
df_train2 = df_dataset.loc[mask]

eixoX_train2 = df_train2[timestamp]
df_train2 = df_train2.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train2], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train2], ignore_index=True)

# Data Test

In [ ]:
start_date = pd.to_datetime("2025-07-01 00:00:00")
end_date = pd.to_datetime("2025-11-30 19:21:00")

mask = (df_dataset[timestamp] > start_date) & (df_dataset[timestamp] <= end_date)
df_test = df_dataset.loc[mask]

eixoX_test = df_test[timestamp]
df_test_aux = df_test.copy()
df_test_aux.set_index(timestamp, inplace=True)
df_test = df_test.drop(timestamp,axis=1)
df_test.reset_index(inplace=True, drop=True)
print(f"Data Test Shape: {df_test.shape}")

# Anomaly Detection

## PCA $\phi$(T$^2$ + Q)

In [ ]:
# Instacinamento da Classe
gain = 2.5
nc = 0
futurai = FuturaiML(nc,gain)
futurai.fit(df_train)

print("T²: {:.2f}".format(futurai.t2_lim))
print("Q: {:.2f}".format(futurai.q_lim))
print("Phi: {:.2f}".format(futurai.phi_lim))
print("Componentes: {:}".format(futurai.nc))

In [ ]:
result = futurai.predict(df_test,eixoX_test)

fig_all_period, _ = utils.dev_graph_predict(result["phi"], result["timestamp"], futurai.phi_lim, " ", start_date, end_date, list_periods=None, plot_anomalies=False)
fig_all_period.show()

## ANN-Based AutoEncoder

Tziolas, T.; Papageorgiou, K.; Theodosiou, T.; Papageorgiou, E.; Mastos, T.; Papadopoulos, A. Autoencoders for Anomaly Detection in an Industrial Multivariate Time Series Dataset. Eng. Proc. 2022, 18, 23. https://doi.org/10.3390/engproc2022018023

In [ ]:
class ANN_Autoencoder(nn.Module):
    def __init__(self, seq_len=60, n_features=3):
        super(ANN_Autoencoder, self).__init__()
        
        self.seq_len = seq_len
        self.n_features = n_features
        
        # Encoder
        # "Input (128-64-32)-flatten-1024-latent space (128)" 
        self.enc_layer1 = nn.Linear(n_features, 128)
        self.enc_layer2 = nn.Linear(128, 64)
        self.enc_layer3 = nn.Linear(64, 32)
        
        # Flatten
        self.flatten_dim = seq_len * 32
        
        self.enc_layer4 = nn.Linear(self.flatten_dim, 1024)
        self.latent_layer = nn.Linear(1024, 128) # Latent Space
        
        # Decoder
        # "(1024-64-32)-reshape (64-128)-output" 
        self.dec_layer1 = nn.Linear(128, 1024)
        self.dec_layer2 = nn.Linear(1024, self.flatten_dim)
        
        # Reshape
        self.dec_layer3 = nn.Linear(32, 64)
        self.dec_layer4 = nn.Linear(64, 128)
        self.output_layer = nn.Linear(128, n_features)
        
        self.relu = nn.ReLU()

    def forward(self, x):
        # x shape: [batch_size, 60, n_features]
        
        # Encoder Pass
        x = self.relu(self.enc_layer1(x))
        x = self.relu(self.enc_layer2(x))
        x = self.relu(self.enc_layer3(x)) # Output: [batch, 60, 32]
        
        # Flatten
        x = x.view(x.size(0), -1)         # Output: [batch, 1920]
        
        x = self.relu(self.enc_layer4(x))
        latent = self.relu(self.latent_layer(x)) # Output: [batch, 128]
        
        # Decoder Pass
        x = self.relu(self.dec_layer1(latent))
        x = self.relu(self.dec_layer2(x)) # Output: [batch, 1920]
        
        # Reshape (unflatten)
        x = x.view(x.size(0), self.seq_len, 32) # Output: [batch, 60, 32]
        
        x = self.relu(self.dec_layer3(x))
        x = self.relu(self.dec_layer4(x))
        
        reconstruction = self.output_layer(x) # Output: [batch, 60, n_features]
        
        return reconstruction

In [ ]:
SEQ_LEN = 60
STRIDE = 1
BATCH_SIZE = 64 # Valor padrão de mercado (32 a 128)
LR = 1e-3
EPOCHS = 1000
PATIENCE = 100    # se não melhorar em 100 épocas para o treino

scaler = MinMaxScaler()
df_train_scaled = scaler.fit_transform(df_train)

def reshape_data(data, seq_len, stride=1):
    """
    (N, features) -> (Num_Janelas, seq_len, features)
    Parâmetros:
      stride (int): passo das janelas 
                    Se stride=1 -> máxima sobreposição
                    Se stride=seq_len -> sem sobreposição
    """
    xs = []
    for i in range(0, len(data) - seq_len + 1, stride):
        x = data[i:(i + seq_len)]
        xs.append(x)
    return np.array(xs)

X_train_full = reshape_data(df_train_scaled, SEQ_LEN, STRIDE)
tensor_data = torch.tensor(X_train_full, dtype=torch.float32)

train_size = int(0.85 * len(tensor_data))

train_tensor = tensor_data[:train_size]
val_tensor = tensor_data[train_size:]

train_dataset = TensorDataset(train_tensor)
val_dataset = TensorDataset(val_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

model_ann = ANN_Autoencoder(seq_len=SEQ_LEN, n_features=df_train.shape[1])
criterion = nn.MSELoss() ## L1Loss || MSELoss
optimizer = optim.Adam(model_ann.parameters(), lr=LR)

best_val_loss = float('inf')
patience_counter = 0

history_loss = []
val_history_loss = []

for epoch in range(EPOCHS):
    model_ann.train()
    train_loss = 0
    for inputs in train_loader:
        if isinstance(inputs, list): inputs = inputs[0]
        optimizer.zero_grad()
        reconstructions = model_ann(inputs)
        loss = criterion(reconstructions, inputs)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    history_loss.append(avg_train_loss)
    
    model_ann.eval()
    val_loss = 0
    with torch.no_grad():
        for inputs in val_loader:
            if isinstance(inputs, list): inputs = inputs[0]
            reconstructions = model_ann(inputs)
            loss = criterion(reconstructions, inputs)
            val_loss += loss.item()
            
    avg_val_loss = val_loss / len(val_loader)
    val_history_loss.append(avg_val_loss)
    
    # Log
    if epoch % 10 == 0:
        print(f"Epoch [{epoch}/{EPOCHS}] | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        # Salvar o melhor modelo
        torch.save(model_ann.state_dict(), 'best_model.pth')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Parada antecipada na época {epoch}. Melhor Val Loss: {best_val_loss:.6f}")
            # Recarregar o melhor modelo
            model_ann.load_state_dict(torch.load('best_model.pth'))
            break

In [ ]:
model_ann.eval()
all_mae_losses = []

with torch.no_grad():
    for batch in train_loader:
        inputs = batch[0]
        
        reconstructions = model_ann(inputs)
        
        # Calcula MAE por amostra -> Result shape: [batch_size]
        batch_loss = torch.mean(torch.abs(reconstructions - inputs), dim=[1, 2])
        
        all_mae_losses.extend(batch_loss.cpu().numpy())
        
train_mae_loss = np.array(all_mae_losses)
THRESHOLD = round(np.mean(train_mae_loss) + (2 * np.std(train_mae_loss)), 2)

print(f"Média do Erro: {np.mean(train_mae_loss):.6f}")
print(f"Desvio Padrão: {np.std(train_mae_loss):.6f}")
print(f"Anomaly Threshold: {THRESHOLD:.2f}")

In [ ]:
def predict(df_test, modelo, scaler, threshold, batch_size=32):
    
    X_test_scaled = scaler.transform(df_test.values)

    seq_len = modelo.seq_len
    sequences = reshape_data(X_test_scaled, SEQ_LEN, STRIDE)
    
    if len(sequences) == 0:
        return np.array([]), np.array([])

    dataset = TensorDataset(torch.tensor(sequences, dtype=torch.float32))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    all_losses = []
    all_preds = []

    modelo.eval()
    with torch.no_grad():
        for batch in loader:
            inputs = batch[0]
            
            # Forward pass
            reconstruction = modelo(inputs)
            
            # MAE por janela
            batch_loss = torch.mean(torch.abs(reconstruction - inputs), dim=[1, 2])
            all_losses.append(batch_loss.numpy())

    final_losses = np.concatenate(all_losses)
    
    return final_losses

erros = predict(df_test, model_ann, scaler, THRESHOLD, batch_size=BATCH_SIZE)

In [ ]:
def plot_anomalies_plotly(df_test, erros, threshold, seq_len=60, stride=1):
    if stride == 1:
        timestamps = df_test.index[seq_len-1:]
    else:
        timestamps = df_test.index[seq_len-1::stride]

    min_len = min(len(timestamps), len(erros))
    timestamps = timestamps[:min_len]
    erros = erros[:min_len]

    plot_df = pd.DataFrame({'erro': erros}, index=timestamps)
    if not isinstance(plot_df.index, pd.DatetimeIndex):
        plot_df.index = pd.to_datetime(plot_df.index)

    freq = f'{stride}min' 
    try:
        plot_df_resampled = plot_df.asfreq(freq)
        plot_df_resampled.fillna(0, inplace=True)
    except ValueError:
        print(" --- Duplicate Indexes ---")
        plot_df_resampled = plot_df

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=plot_df_resampled.index,
        y=plot_df_resampled['erro'],
        mode='lines',
        name='Erro de Reconstrução',
        fill="tozeroy", 
        line=dict(color="#0F293A"),
        connectgaps=False
    ))
    
    fig.add_hline(
        y=threshold, 
        name='Limiar',
        line_dash="dash", 
        line_color="red", 
        line_width=2
    )
    
    fig.update_layout(
        xaxis_title='Data/Hora',
        yaxis_title='Erro Médio Absoluto (MAE)',
        template='plotly_white',
        hovermode="x unified",
        showlegend=True
    )

    fig.show()
    
plot_anomalies_plotly(df_test_aux, erros, THRESHOLD, seq_len=60, stride=1)

## LSTM-Based AutoEncoder

Tziolas, T.; Papageorgiou, K.; Theodosiou, T.; Papageorgiou, E.; Mastos, T.; Papadopoulos, A. Autoencoders for Anomaly Detection in an Industrial Multivariate Time Series Dataset. Eng. Proc. 2022, 18, 23. https://doi.org/10.3390/engproc2022018023

In [ ]:
class LSTM_Autoencoder(nn.Module):
    def __init__(self, seq_len=60, n_features=3, hidden_dim=128):
        super(LSTM_Autoencoder, self).__init__()
        
        self.seq_len = seq_len
        self.n_features = n_features
        self.hidden_dim = hidden_dim
        
        # --- Encoder ---
        # Input: [batch, 60, 3] -> Output: [batch, 60, 128]
        self.encoder_l1 = nn.LSTM(
            input_size=n_features, 
            hidden_size=hidden_dim, 
            batch_first=True
        )
        
        # Input: [batch, 60, 128] -> Output (Last Hidden): [batch, 128]
        self.encoder_l2 = nn.LSTM(
            input_size=hidden_dim, 
            hidden_size=hidden_dim, 
            batch_first=True
        )
        
        # --- Decoder ---
        # Input: [batch, 60, 128] -> Output: [batch, 60, 128]
        self.decoder_l1 = nn.LSTM(
            input_size=hidden_dim, 
            hidden_size=hidden_dim, 
            batch_first=True
        )
        
        # Camada 2 (Saída): Reconstrói as features originais
        # Input: [batch, 60, 128] -> Output: [batch, 60, 3]
        self.decoder_l2 = nn.LSTM(
            input_size=hidden_dim, 
            hidden_size=n_features, 
            batch_first=True
        )

    def forward(self, x):
        # x shape: [batch, 60, 3]
        
        # Encoder Pass
        # save enc_out1 for Skip Connection
        enc_out1, _ = self.encoder_l1(x) # shape: [batch, 60, 128]
        
        # Segunda camada LSTM apenas o último estado oculto
        _, (hidden, _) = self.encoder_l2(enc_out1)
        latent = hidden[-1] # shape: [batch, 128]
        
        # Repeat Vector
        # Repete o vetor latente 60 vezes para corresponder a sequência temporal
        # [batch, 128] -> [batch, 60, 128]
        repeat_vect = latent.unsqueeze(1).repeat(1, self.seq_len, 1)
        
        # Skip Connection
        decoder_input = enc_out1 + repeat_vect 
        
        # Decoder Pass
        dec_out1, _ = self.decoder_l1(decoder_input)
        
        # Camada final de reconstrução
        reconstruction, _ = self.decoder_l2(dec_out1)
        
        return reconstruction

In [ ]:
SEQ_LEN = 60
STRIDE = 1
BATCH_SIZE = 7807
LR = 1e-3
EPOCHS = 1000

model_lstm = LSTM_Autoencoder(seq_len=SEQ_LEN, n_features=df_train.shape[1])

## CNN-Based AutoEncoder

Tziolas, T.; Papageorgiou, K.; Theodosiou, T.; Papageorgiou, E.; Mastos, T.; Papadopoulos, A. Autoencoders for Anomaly Detection in an Industrial Multivariate Time Series Dataset. Eng. Proc. 2022, 18, 23. https://doi.org/10.3390/engproc2022018023

## USAD

Audibert, J., Michiardi, P., Guyard, F., Marti, S., Zuluaga, M. A. (2020).
USAD : UnSupervised Anomaly Detection on multivariate time series.
Proceedings of the 26th ACM SIGKDD International Conference on Knowledge Discovery & Data Mining, August 23-27, 2020

## OmniAnomaly

Ya Su, Youjian Zhao, Chenhao Niu, Rong Liu, Wei Sun, and Dan Pei. 2019. Robust Anomaly Detection for Multivariate Time Series through Stochastic Recurrent Neural Network. In Proceedings of the 25th ACM SIGKDD International Conference on Knowledge Discovery & Data Mining (KDD '19). Association for Computing Machinery, New York, NY, USA, 2828–2837. https://doi.org/10.1145/3292500.3330672

## MSCRED

Chuxu Zhang, Dongjin Song, Yuncong Chen, Xinyang Feng, Cristian Lumezanu, Wei Cheng, Jingchao Ni, Bo Zong, Haifeng Chen, and Nitesh V. Chawla. 2019. A deep neural network for unsupervised anomaly detection and diagnosis in multivariate time series data. In Proceedings of the Thirty-Third AAAI Conference on Artificial Intelligence and Thirty-First Innovative Applications of Artificial Intelligence Conference and Ninth AAAI Symposium on Educational Advances in Artificial Intelligence (AAAI'19/IAAI'19/EAAI'19), Vol. 33. AAAI Press, Article 174, 1409–1416. https://doi.org/10.1609/aaai.v33i01.33011409